In [1]:
import pandas as pd

# Load the CSV file
file_path = "cis.csv"  # Change this if needed
df = pd.read_csv(file_path)

# Display the first few rows
df.head()


,Cost_Rank,Product_Name,Product_Life_Cycle,FY22_Q2,FY22_Q3,FY22_Q4,FY23_Q1,FY23_Q2,FY23_Q3,FY23_Q4,...,M_FY2025_Q1_BIAS,S_FY2024_Q3_Accuracy,S_FY2024_Q3_BIAS,S_FY2024_Q4_Accuracy,S_FY2024_Q4_BIAS,S_FY2025_Q1_Accuracy,S_FY2025_Q1_BIAS,D_FY2025_Q2_Forecasted,M_FY2025_Q2_Forecasted,S_FY2025_Q2_Forecasted
0,1,SWITCH Enterprise High 1,Sustaining,10395,10400,6592,10132,6809,6928,10299,...,2.51%,76.47%,-23.53%,80.84%,-19.16%,90.63%,-9.37%,9575,9523,8158
1,2,ACCESS POINT Mid 1,Decline,131048,90440,99442,67630,84349,67670,55237,...,1.11%,88.53%,11.47%,69.21%,-30.79%,94.25%,5.75%,61871,55778,54959
2,3,SWITCH Enterprise Mid 1,Sustaining,19897,15482,13128,8770,6703,7056,8138,...,29.09%,87.55%,-12.45%,96.72%,3.28%,97.45%,-2.55%,8317,8012,8235
3,4,SWITCH Enterprise Mid 2,Sustaining,18638,27319,18233,11190,10927,8358,11681,...,95.22%,95.36%,-4.64%,90.61%,9.39%,71.62%,28.38%,10445,11239,9618
4,5,SWITCH Data Center Low 1,Sustaining,6629,4275,2981,2956,1987,1709,2326,...,-24.27%,63.55%,-36.45%,79.11%,-20.89%,98.53%,-1.47%,2550,2283,2496


In [3]:
# Convert percentage strings to numerical values for accuracy and bias columns
percentage_columns = [col for col in df.columns if "Accuracy" in col or "BIAS" in col]

# Remove '%' and convert to float
for col in percentage_columns:
    df[col] = df[col].str.rstrip('%').astype(float) / 100

# Check cleaned data
df.info()
df.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 36 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Cost_Rank               10 non-null     int64  
 1   Product_Name            10 non-null     object 
 2   Product_Life_Cycle      10 non-null     object 
 3   FY22_Q2                 10 non-null     int64  
 4   FY22_Q3                 10 non-null     int64  
 5   FY22_Q4                 10 non-null     int64  
 6   FY23_Q1                 10 non-null     int64  
 7   FY23_Q2                 10 non-null     int64  
 8   FY23_Q3                 10 non-null     int64  
 9   FY23_Q4                 10 non-null     int64  
 10  FY24_Q1                 10 non-null     int64  
 11  FY24_Q2                 10 non-null     int64  
 12  FY24_Q3                 10 non-null     int64  
 13  FY24_Q4                 10 non-null     int64  
 14  FY25_Q1                 10 non-null     int64

,Cost_Rank,Product_Name,Product_Life_Cycle,FY22_Q2,FY22_Q3,FY22_Q4,FY23_Q1,FY23_Q2,FY23_Q3,FY23_Q4,...,M_FY2025_Q1_BIAS,S_FY2024_Q3_Accuracy,S_FY2024_Q3_BIAS,S_FY2024_Q4_Accuracy,S_FY2024_Q4_BIAS,S_FY2025_Q1_Accuracy,S_FY2025_Q1_BIAS,D_FY2025_Q2_Forecasted,M_FY2025_Q2_Forecasted,S_FY2025_Q2_Forecasted
0,1,SWITCH Enterprise High 1,Sustaining,10395,10400,6592,10132,6809,6928,10299,...,0.0251,0.7647,-0.2353,0.8084,-0.1916,0.9063,-0.0937,9575,9523,8158
1,2,ACCESS POINT Mid 1,Decline,131048,90440,99442,67630,84349,67670,55237,...,0.0111,0.8853,0.1147,0.6921,-0.3079,0.9425,0.0575,61871,55778,54959
2,3,SWITCH Enterprise Mid 1,Sustaining,19897,15482,13128,8770,6703,7056,8138,...,0.2909,0.8755,-0.1245,0.9672,0.0328,0.9745,-0.0255,8317,8012,8235
3,4,SWITCH Enterprise Mid 2,Sustaining,18638,27319,18233,11190,10927,8358,11681,...,0.9522,0.9536,-0.0464,0.9061,0.0939,0.7162,0.2838,10445,11239,9618
4,5,SWITCH Data Center Low 1,Sustaining,6629,4275,2981,2956,1987,1709,2326,...,-0.2427,0.6355,-0.3645,0.7911,-0.2089,0.9853,-0.0147,2550,2283,2496


In [5]:
# Function to compute forecasted values based on actual sales and bias
def compute_forecasted_values(actual, bias):
    return actual * (1 + bias)

# Define the exact column names from your dataset
actual_quarters = ['FY24_Q3', 'FY24_Q4', 'FY25_Q1']
bias_quarters = ['FY2024_Q3', 'FY2024_Q4', 'FY2025_Q1']  # Adjust if needed

teams = ['D', 'M', 'S']

# Compute forecasted values for each team and each quarter
for team in teams:
    for i in range(len(actual_quarters)):
        actual_col = actual_quarters[i]  # Actual sales column
        bias_col = f"{team}_{bias_quarters[i]}_BIAS"  # Bias column for each team
        forecast_col = f"{team}_{actual_quarters[i]}_Forecasted"  # New column to store forecasted values

        # Check if both actual sales and bias columns exist before calculation
        if actual_col in df.columns and bias_col in df.columns:
            df[forecast_col] = compute_forecasted_values(df[actual_col], df[bias_col])
        else:
            print(f"Skipping {forecast_col} because {actual_col} or {bias_col} is missing.")

# Display the first few rows to verify forecasted values
df[[col for col in df.columns if 'Forecasted' in col]].head()


,D_FY2025_Q2_Forecasted,M_FY2025_Q2_Forecasted,S_FY2025_Q2_Forecasted,D_FY24_Q3_Forecasted,D_FY24_Q4_Forecasted,D_FY25_Q1_Forecasted,M_FY24_Q3_Forecasted,M_FY24_Q4_Forecasted,M_FY25_Q1_Forecasted,S_FY24_Q3_Forecasted,S_FY24_Q4_Forecasted,S_FY25_Q1_Forecasted
0,9575,9523,8158,8100.0363,10362.1530,9291.1770,9727.0842,11551.2355,9788.6799,8412.4647,8395.2340,8654.2587
1,61871,55778,54959,55649.0407,61525.4800,55060.6545,92662.0995,65694.6800,53836.0195,57720.2807,41221.4760,56306.5875
2,8317,8012,8235,7602.2280,10289.7600,9323.5962,7701.4302,10897.9500,11775.5898,8599.1610,10018.1600,8889.3890
3,10445,11239,9618,11861.1975,12871.1186,10897.7680,10802.7090,15286.0074,16517.5642,10818.5920,10879.9294,10862.2318
4,2550,2283,2496,2099.8624,2300.1360,2639.8880,1987.6512,2212.8784,1797.8302,2003.0960,2537.8488,2339.1022


In [7]:
# Print all column names to check for mismatches
print(df.columns.tolist())


['Cost_Rank', 'Product_Name', 'Product_Life_Cycle', 'FY22_Q2', 'FY22_Q3', 'FY22_Q4', 'FY23_Q1', 'FY23_Q2', 'FY23_Q3', 'FY23_Q4', 'FY24_Q1', 'FY24_Q2', 'FY24_Q3', 'FY24_Q4', 'FY25_Q1', 'D_FY2024_Q3_Accuracy', 'D_FY2024_Q3_BIAS', 'D_FY2024_Q4_Accuracy', 'D_FY2024_Q4_BIAS', 'D_FY2025_Q1_Accuracy', 'D_FY2025_Q1_BIAS', 'M_FY2024_Q3_Accuracy', 'M_FY2024_Q3_BIAS', 'M_FY2024_Q4_Accuracy', 'M_FY2024_Q4_BIAS', 'M_FY2025_Q1_Accuracy', 'M_FY2025_Q1_BIAS', 'S_FY2024_Q3_Accuracy', 'S_FY2024_Q3_BIAS', 'S_FY2024_Q4_Accuracy', 'S_FY2024_Q4_BIAS', 'S_FY2025_Q1_Accuracy', 'S_FY2025_Q1_BIAS', 'D_FY2025_Q2_Forecasted', 'M_FY2025_Q2_Forecasted', 'S_FY2025_Q2_Forecasted', 'D_FY24_Q3_Forecasted', 'D_FY24_Q4_Forecasted', 'D_FY25_Q1_Forecasted', 'M_FY24_Q3_Forecasted', 'M_FY24_Q4_Forecasted', 'M_FY25_Q1_Forecasted', 'S_FY24_Q3_Forecasted', 'S_FY24_Q4_Forecasted', 'S_FY25_Q1_Forecasted']


In [9]:
df.to_csv("forecasted_values.csv", index=False)


In [11]:
# Dictionary to store the best team for each product
best_teams_per_product = {}

# Loop through each product (row-wise operation)
for idx, row in df.iterrows():
    team_accuracies = {
        "D": row[["D_FY2024_Q3_Accuracy", "D_FY2024_Q4_Accuracy", "D_FY2025_Q1_Accuracy"]].mean(),
        "M": row[["M_FY2024_Q3_Accuracy", "M_FY2024_Q4_Accuracy", "M_FY2025_Q1_Accuracy"]].mean(),
        "S": row[["S_FY2024_Q3_Accuracy", "S_FY2024_Q4_Accuracy", "S_FY2025_Q1_Accuracy"]].mean(),
    }
    
    # Find the team with the highest average accuracy
    best_team = max(team_accuracies, key=team_accuracies.get)
    
    # Store the best team and their accuracy
    best_teams_per_product[df.loc[idx, "Product_Name"]] = (best_team, team_accuracies[best_team])

# Print results
print("Best forecasting team per product:\n")
for product, (team, accuracy) in best_teams_per_product.items():
    print(f"{product}: Best Team = {team}, Accuracy = {accuracy:.4f}")


Best forecasting team per product:

SWITCH Enterprise High 1: Best Team = M, Accuracy = 0.9156
ACCESS POINT Mid 1: Best Team = D, Accuracy = 0.9527
SWITCH Enterprise Mid 1: Best Team = S, Accuracy = 0.9391
SWITCH Enterprise Mid 2: Best Team = S, Accuracy = 0.8586
SWITCH Data Center Low 1: Best Team = S, Accuracy = 0.8040
SWITCH Enterprise High 2: Best Team = M, Accuracy = 0.7572
SWITCH Data Center Low 2: Best Team = S, Accuracy = 0.7010
Wireless Controller 1: Best Team = D, Accuracy = 0.9217
ACCESS POINT Mid 2: Best Team = S, Accuracy = 0.9081
Wireless Controller 2: Best Team = S, Accuracy = 0.8665


In [13]:
# Initialize dictionaries to store results
best_team_per_product = {}

# Loop through each product
for idx, row in df.iterrows():
    # Compute average accuracy for each team
    avg_accuracy = {
        "D": row[["D_FY2024_Q3_Accuracy", "D_FY2024_Q4_Accuracy", "D_FY2025_Q1_Accuracy"]].mean(),
        "M": row[["M_FY2024_Q3_Accuracy", "M_FY2024_Q4_Accuracy", "M_FY2025_Q1_Accuracy"]].mean(),
        "S": row[["S_FY2024_Q3_Accuracy", "S_FY2024_Q4_Accuracy", "S_FY2025_Q1_Accuracy"]].mean(),
    }

    # Compute average bias for each team
    avg_bias = {
        "D": row[["D_FY2024_Q3_BIAS", "D_FY2024_Q4_BIAS", "D_FY2025_Q1_BIAS"]].mean(),
        "M": row[["M_FY2024_Q3_BIAS", "M_FY2024_Q4_BIAS", "M_FY2025_Q1_BIAS"]].mean(),
        "S": row[["S_FY2024_Q3_BIAS", "S_FY2024_Q4_BIAS", "S_FY2025_Q1_BIAS"]].mean(),
    }

    # Find the team with the highest accuracy
    best_team = max(avg_accuracy, key=avg_accuracy.get)
    
    # Store results
    best_team_per_product[row["Product_Name"]] = {
        "Best Team": best_team,
        "Accuracy": avg_accuracy[best_team],
        "Bias": avg_bias[best_team]
    }

# Convert to DataFrame for better readability
best_team_df = pd.DataFrame.from_dict(best_team_per_product, orient="index")

# Print best forecasting team for each product
print(best_team_df)


                         Best Team  Accuracy      Bias
SWITCH Enterprise High 1         M  0.915600  0.007200
ACCESS POINT Mid 1               D  0.952733  0.047267
SWITCH Enterprise Mid 1          S  0.939067 -0.039067
SWITCH Enterprise Mid 2          S  0.858633  0.110433
SWITCH Data Center Low 1         S  0.803967 -0.196033
SWITCH Enterprise High 2         M  0.757167  0.038233
SWITCH Data Center Low 2         S  0.700967  0.112300
Wireless Controller 1            D  0.921700  0.002233
ACCESS POINT Mid 2               S  0.908133  0.073533
Wireless Controller 2            S  0.866500  0.133500
